# ProChem core functionality demo

Этот notebook показывает текущий уровень проработки ядра `prochem`: модели данных, VASP I/O, склейку restart-расчетов, dataset-режим для MLIP, анализ траекторий и экспорт таблиц.

Все пути ниже можно менять под локальные расчеты. Notebook рассчитан на запуск из корня репозитория `B:\\Science\\prochem`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT, SRC

In [ ]:
import numpy as np
import pandas as pd

from prochem.core import Calculation, Structure, Trajectory, TrajectoryMergePolicy
from prochem.io import parse
from prochem.io.vasp import (
    discover_structure_files,
    discover_vasprun_files,
    parse_structure_dataset,
    parse_structure_dataset_as_trajectory,
    parse_vasprun_sequence,
)
from prochem.analysis import (
    add_distance_columns,
    add_kinetic_energy_columns,
    add_velocity_columns,
    center_of_mass_dataframe,
    coordinate_dataframe,
    export_dataframe,
)

print("imports ok")

## Тестовые пути

`MAIN_VASPRUN` - одиночный `vasprun.xml`.
`DELETION_RESTART_DIR` - пример restart, где при продолжении удалялся атом.
`EXACT_RESTART_DIR` - пример restart с обычными exact overlap-сегментами.

In [ ]:
MAIN_VASPRUN = Path(r"B:\Science\Calculations\VASP\Low-k\Poss_with_Ar\POSS_Ar_30_grad_20eV\vasprun.xml")
DELETION_RESTART_DIR = Path(r"B:\Science\Calculations\VASP\ALE\C12F26\Ar\C\30eV")
EXACT_RESTART_DIR = Path(r"B:\Science\Calculations\VASP\MoS2\N2\Mo\MoS2_N2_30eV_parallel_Mo")

for path in [MAIN_VASPRUN, DELETION_RESTART_DIR, EXACT_RESTART_DIR]:
    print(path, "exists=", path.exists())

## Базовые модели: Calculation, Trajectory, Structure

`parse()` выбирает парсер через `prochem.io.registry`. Для VASP возвращается единая модель `Calculation`.

In [ ]:
calculation = parse(MAIN_VASPRUN)
trajectory = calculation.trajectory
first = trajectory.frame(0)

print(type(calculation).__name__, calculation.engine, calculation.source)
print("steps:", calculation.step_count)
print("registry atoms:", calculation.atom_count)
print("first frame atoms:", first.atom_count)
print("cell:\n", first.cell.vectors)
print("first species:", first.species[:10])

## Плотные массивы траектории

`Trajectory` хранит список `Structure`, но умеет отдавать плотные массивы `(steps, registry_atoms, 3)`. Если атом был удален при restart, его слоты заполняются `NaN`.

In [ ]:
positions = trajectory.positions_array()
direct = trajectory.direct_positions_array()
mask = trajectory.presence_mask()

print("positions:", positions.shape)
print("direct:", direct.shape)
print("presence mask:", mask.shape)
print("missing atom slots:", int((~mask).sum()))

## Склейка restart-расчетов

`TrajectoryMergePolicy.boundary_search_frames` ищет overlap не только в первом кадре следующего `vasprun`, но и в первых N кадрах. Это нужно для случая, когда POSCAR был сделан из CONTCAR с оставленным блоком после скоростей: VASP может дать один или несколько стартовых кадров, которые не совпадают с концом предыдущего сегмента.

`allow_mismatch_fallback=False` запрещает fallback, при котором несовпадающий сегмент регистрируется как новые атомы. Это полезно для строгой проверки restart-склейки и для MLIP dataset workflow.

In [ ]:
policy = TrajectoryMergePolicy(
    boundary_search_frames=6,
    allow_mismatch_fallback=False,
    keep_topology_change_frame=True,
)

merged, report = parse_vasprun_sequence(DELETION_RESTART_DIR, policy=policy)
for event in report.events:
    print(
        event.status,
        "next=", event.next_source.name,
        "matched=", event.matched_next_frame,
        "drop=", event.dropped_next_frames,
        "deleted=", event.deleted_atom_ids,
        "max_delta=", event.max_delta,
    )

print("merged steps:", merged.step_count)
print("registry atoms:", merged.atom_count)
print("missing slots:", int((~merged.trajectory.presence_mask()).sum()))

## Analysis: координаты, скорости, энергии, расстояния

Функции из `prochem.analysis` не зависят от GUI. Их можно использовать из Qt, Jupyter и web API.

In [ ]:
traj = merged.trajectory
atom_ids = [record.atom_id for record in traj.atom_registry[:3]]

df = coordinate_dataframe(traj, atom_ids=atom_ids)
df = add_velocity_columns(df, traj, atom_ids)
try:
    df = add_kinetic_energy_columns(df, traj, atom_ids)
except ValueError as exc:
    print("kinetic energy skipped:", exc)

df = add_distance_columns(df, traj, [(atom_ids[0], atom_ids[1])])
df.head()

In [ ]:
cm = center_of_mass_dataframe(traj, atom_ids, name="cm_first3")
cm.head()

## MLIP dataset mode

Для датасетов структуры считаются независимыми конфигурациями, а не restart-сегментами. Поэтому используется `discover_structure_files()` / `parse_structure_dataset()` / `parse_structure_dataset_as_trajectory()`.

По умолчанию на каждую подпапку выбирается `CONTCAR`, а если его нет - `POSCAR`. При упаковке в одну trajectory включен `strict_topology=True`: если атомный состав или порядок видов отличается, функция падает вместо fallback-регистрации новых атомов.

In [ ]:
structure_files = discover_structure_files(DELETION_RESTART_DIR, recursive=True)
print("found structure files:", len(structure_files))
for path in structure_files[:10]:
    print(path)

In [ ]:
# Пример упаковки dataset в trajectory. Раскомментируйте на директории,
# где все структуры имеют одинаковую топологию и порядок атомов.
# dataset_calc = parse_structure_dataset_as_trajectory(DELETION_RESTART_DIR, recursive=True)
# print(dataset_calc.step_count, dataset_calc.atom_count)
# dataset_calc.trajectory.frame(0).properties

## Export

`export_dataframe()` умеет писать `.csv`, `.xlsx`, `.html`. Ниже пример CSV в локальную папку `notebooks/output`.

In [ ]:
output_dir = ROOT / "notebooks" / "output"
output_dir.mkdir(exist_ok=True)
output_path = export_dataframe(df.head(100), output_dir / "core_demo_table.csv")
output_path

## Что уже проработано и где вероятны правки

- Готово: единые `Calculation/Trajectory/Structure`, VASP parser registry, restart-склейка, удаление атомов между restart, плотные массивы с `NaN`, базовые таблицы и экспорт.
- Добавлено сейчас: поиск overlap в первых N кадрах следующего `vasprun`, no-fallback режим, dataset-сборка POSCAR/CONTCAR из подпапок.
- Требует будущих правок: полноценный `StructureDataset` как отдельная core-модель, нормальная сериализация больших массивов в HDF5/Zarr, независимые rendering DTO из trajectory, Jupyter/Plotly визуализация, web schemas на Pydantic.